# Notebook 5 — Live Inference Pipeline

**This is the user-facing notebook.** Point it at any Wireshark `.csv` export and it will:
1. Load and validate the capture file
2. Extract the same flow-level features used during training
3. Run the trained model to classify each flow
4. Output a per-flow results table and a visual summary

## How to export from Wireshark
1. Capture your traffic in Wireshark
2. Go to **File → Export Packet Dissections → As CSV...**
3. Make sure these columns are included: `No.`, `Time`, `Source`, `Destination`, `Protocol`, `Length`, `Info`
4. Set `CAPTURE_CSV` below to the path of your exported file

In [ ]:
import sys
import json
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

sys.path.insert(0, str(Path('.').resolve()))
from utils.feature_engineering import (
    preprocess_packets,
    extract_flow_features,
    FEATURE_COLS,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

MODELS_DIR = Path('models')

# ─────────────────────────────────────────────────────────────────
# ↓ SET THIS to your Wireshark CSV export path
CAPTURE_CSV = Path('your_capture.csv')
# ─────────────────────────────────────────────────────────────────

print('Inference pipeline ready.')
print(f'Capture file: {CAPTURE_CSV}')

## 1. Load trained model artifacts

In [ ]:
model    = joblib.load(MODELS_DIR / 'best_model.pkl')
scaler   = joblib.load(MODELS_DIR / 'scaler.pkl')
le       = joblib.load(MODELS_DIR / 'label_encoder.pkl')

with open(MODELS_DIR / 'model_metadata.json') as f:
    meta = json.load(f)

print(f'Model      : {meta["best_model"]}')
print(f'Accuracy   : {meta["best_accuracy"]*100:.2f}%')
print(f'Classes    : {meta["classes"]}')
print(f'Features   : {FEATURE_COLS}')

## 2. Load & validate Wireshark CSV

In [ ]:
REQUIRED_COLS = {'No.', 'Time', 'Source', 'Destination', 'Protocol', 'Length', 'Info'}

def load_wireshark_csv(path: Path) -> pd.DataFrame:
    """Load a Wireshark CSV export with fallback encodings and column validation."""
    if not path.exists():
        raise FileNotFoundError(
            f'File not found: {path}\n'
            'Export from Wireshark via: File → Export Packet Dissections → As CSV'
        )

    # Try common encodings
    df = None
    for enc in ['utf-8', 'latin-1', 'cp1252']:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False)
            break
        except Exception:
            continue

    if df is None:
        raise ValueError(f'Could not read {path} with any encoding.')

    # Validate columns (case-insensitive)
    found = {c.strip() for c in df.columns}
    # Try to match even if the column names differ slightly
    missing = [r for r in REQUIRED_COLS
               if not any(r.lower() in f.lower() for f in found)]
    if missing:
        raise ValueError(
            f'Missing expected columns: {missing}\n'
            f'Found columns: {list(df.columns)}\n'
            'Make sure to export with all columns from Wireshark.'
        )

    return df


raw_capture = load_wireshark_csv(CAPTURE_CSV)
print(f'Loaded capture: {len(raw_capture):,} packets')
print(f'Columns: {raw_capture.columns.tolist()}')
raw_capture.head(5)

## 3. Preprocess & extract flow features

Uses **identical** logic to Notebook 3 via `utils/feature_engineering.py`.

In [ ]:
print('Step 1/3: Preprocessing packets...')
preprocessed = preprocess_packets(raw_capture)
print(f'  {len(preprocessed):,} packets after cleaning')

print('Step 2/3: Extracting flow features...')
# No label_col — inference mode
flows = extract_flow_features(preprocessed, label_col=None)
print(f'  {len(flows):,} flows extracted')

print('Step 3/3: Scaling features...')
X_live = flows[FEATURE_COLS].copy()
X_live_scaled = scaler.transform(X_live)
print('  Done!')

## 4. Run predictions

In [ ]:
# Predict class and probability for each flow
y_pred_encoded  = model.predict(X_live_scaled)
y_pred_labels   = le.inverse_transform(y_pred_encoded)
y_pred_proba    = model.predict_proba(X_live_scaled)

# Confidence = probability of predicted class
confidence = y_pred_proba.max(axis=1)

# Attach predictions to flow DataFrame
results = flows.copy()
results['prediction']  = y_pred_labels
results['confidence']  = (confidence * 100).round(1)

print(f'Classified {len(results):,} flows')
results[['flow_id', 'packet_count', 'duration', 'packets_per_second',
          'prediction', 'confidence']].head(10)

## 5. Summary report

In [ ]:
summary = (
    results.groupby('prediction')
    .agg(
        flow_count     = ('flow_id', 'count'),
        avg_confidence = ('confidence', 'mean'),
        avg_packets    = ('packet_count', 'mean'),
    )
    .reset_index()
)
summary['pct_of_flows'] = (summary['flow_count'] / len(results) * 100).round(2)
summary = summary.sort_values('flow_count', ascending=False)

print('=' * 60)
print('  NETWORK ANOMALY DETECTION — ANALYSIS SUMMARY')
print('=' * 60)
print(f'  Capture file : {CAPTURE_CSV.name}')
print(f'  Total packets: {len(raw_capture):,}')
print(f'  Total flows  : {len(results):,}')
print()
print(summary.to_string(index=False))
print('=' * 60)

# Highlight if anomalies found
benign_pct = summary.loc[summary['prediction'] == 'Benign', 'pct_of_flows'].sum()
anomaly_pct = 100 - benign_pct
if anomaly_pct > 0:
    print(f'\n⚠️  {anomaly_pct:.1f}% of flows classified as ANOMALOUS traffic!')
else:
    print('\n✅ All flows classified as Benign.')

## 6. Visualise results

In [ ]:
# Colour map: Benign = green, everything else = shades of red/orange
all_classes = meta['classes']
color_map = {
    'Benign':      '#4CAF50',
    'Flood':       '#F44336',
    'Malware':     '#9C27B0',
    'Brute_Force': '#FF9800',
    'Exploit':     '#F50057',
    'Probe':       '#2196F3',
}
default_color = '#607D8B'

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar chart: flow counts ---
bars = axes[0].bar(
    summary['prediction'],
    summary['flow_count'],
    color=[color_map.get(p, default_color) for p in summary['prediction']]
)
axes[0].set_title('Detected flow types', fontsize=13)
axes[0].set_xlabel('Prediction')
axes[0].set_ylabel('Number of flows')
axes[0].tick_params(axis='x', rotation=20)
for bar in bars:
    axes[0].annotate(
        f'{int(bar.get_height())}',
        (bar.get_x() + bar.get_width()/2, bar.get_height()),
        ha='center', va='bottom', fontsize=10
    )

# --- Pie chart: percentage breakdown ---
pie_colors = [color_map.get(p, default_color) for p in summary['prediction']]
axes[1].pie(
    summary['flow_count'],
    labels=summary['prediction'],
    autopct='%1.1f%%',
    colors=pie_colors,
    startangle=140
)
axes[1].set_title('Traffic composition', fontsize=13)

plt.suptitle(
    f'Wireshark Capture Analysis — {CAPTURE_CSV.name}',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(f'analysis_{CAPTURE_CSV.stem}.png', dpi=150)
plt.show()
print(f'Chart saved as: analysis_{CAPTURE_CSV.stem}.png')

## 7. Per-flow detail table

Full breakdown of every flow with its prediction and confidence score.

In [ ]:
detail_cols = [
    'flow_id', 'packet_count', 'duration', 'avg_length',
    'packets_per_second', 'bytes_per_second',
    'syn_count', 'ack_count', 'http_request_count',
    'prediction', 'confidence'
]

detail = results[detail_cols].sort_values('confidence', ascending=False)
detail = detail.round(3)

# Save to CSV for further analysis
out_csv = f'results_{CAPTURE_CSV.stem}.csv'
detail.to_csv(out_csv, index=False)
print(f'Per-flow results saved to: {out_csv}')
print(f'\nTop 10 highest-confidence flows:')
detail.head(10)

## 8. Suspicious flows deep-dive

Filter to show only non-Benign flows for investigation.

In [ ]:
suspicious = detail[detail['prediction'] != 'Benign']

if len(suspicious) == 0:
    print('✅ No suspicious flows detected in this capture.')
else:
    print(f'⚠️  {len(suspicious)} suspicious flows detected:\n')
    print(suspicious.to_string(index=False))

## ✅ Inference complete!

**Outputs:**
- `analysis_<filename>.png` — visual breakdown chart
- `results_<filename>.csv` — per-flow results table

**Interpreting results:**
| Class | What it means |
|---|---|
| `Benign` | Normal network traffic |
| `Flood` | Volume-based DoS attack (SYN/UDP/HTTP/ICMP flood) |
| `Malware` | Malicious software communication (ransomware, trojan, spyware) |
| `Brute_Force` | Repeated login/auth attempts |
| `Exploit` | Vulnerability exploitation attempt |
| `Probe` | Network scanning / reconnaissance |